# ML Interview Reference Notebook (From Scratch + PyTorch)
This notebook is designed for **fast recall + implementation** in ML coding interviews.

**Style goals**
- Small, reusable helper functions
- Each algorithm has: **fit**, **predict**, and a **sanity-check demo**
- Focus on correctness, shape discipline, and numerically-stable basics

> Tip: In interviews, start with shapes + objective, then implement the cleanest baseline.

## 0) Setup & Utilities

In [2]:
import numpy as np

# Reproducibility
def seed_all(seed=42):
    np.random.seed(seed)

seed_all(42)

def train_test_split(X, y, test_size=0.2, seed=42):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    idx = np.arange(n)
    rng.shuffle(idx)
    n_test = int(n * test_size)
    test_idx = idx[:n_test]
    train_idx = idx[n_test:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

def standardize_fit(X):
    mu = X.mean(axis=0, keepdims=True)
    sigma = X.std(axis=0, keepdims=True) + 1e-12
    return mu, sigma

def standardize_transform(X, mu, sigma):
    return (X - mu) / sigma

def add_bias(X):
    # Appends a bias feature of 1s: (n,d) -> (n,d+1)
    return np.concatenate([X, np.ones((X.shape[0], 1))], axis=1)

def mse(y_hat, y):
    y_hat = y_hat.reshape(-1)
    y = y.reshape(-1)
    return np.mean((y_hat - y)**2)

def accuracy(y_hat, y):
    y_hat = y_hat.reshape(-1)
    y = y.reshape(-1)
    return np.mean(y_hat == y)

def sigmoid(z):
    # numerically-stable sigmoid
    z = np.clip(z, -60, 60)
    return 1.0 / (1.0 + np.exp(-z))

def log_loss(y_prob, y):
    # Binary cross-entropy
    y_prob = np.clip(y_prob, 1e-12, 1 - 1e-12)
    return -np.mean(y * np.log(y_prob) + (1 - y) * np.log(1 - y_prob))

def softmax(Z):
    # stable softmax across last axis
    Z = Z - np.max(Z, axis=1, keepdims=True)
    expZ = np.exp(Z)
    return expZ / np.sum(expZ, axis=1, keepdims=True)

## 1) Linear Regression (NumPy) — Closed Form + Gradient Descent
**Objective**: minimize \( \|Xw - y\|^2 \) (with optional bias).

In [4]:
def linreg_closed_form(X, y, l2=0.0):
    # X: (n,d), y: (n,) or (n,1). Assumes bias already included if desired.
    y = y.reshape(-1, 1)
    d = X.shape[1]
    A = X.T @ X + l2 * np.eye(d)
    b = X.T @ y
    w = np.linalg.solve(A, b)  # (d,1)
    return w.reshape(-1)

def linreg_gd_fit(X, y, lr=0.05, steps=2000, l2=0.0):
    # Adds bias inside by convention (so caller doesn't forget)
    Xb = add_bias(X)
    n, d = Xb.shape
    y = y.reshape(-1)
    w = np.zeros(d)

    for _ in range(steps):
        y_hat = Xb @ w
        grad = (2.0 / n) * (Xb.T @ (y_hat - y)) + 2*l2*w
        w -= lr * grad
    return w  # includes bias as last element

def linreg_predict(X, w):
    Xb = add_bias(X)
    return Xb @ w

# Demo
seed_all(1)
n, d = 300, 3
X = np.random.randn(n, d)
true_w = np.array([2.0, -3.0, 0.5])
true_b = 1.2
y = X @ true_w + true_b + 0.3*np.random.randn(n)

mu, sigma = standardize_fit(X)
Xs = standardize_transform(X, mu, sigma)

w_gd = linreg_gd_fit(Xs, y, lr=0.1, steps=3000)
y_hat = linreg_predict(Xs, w_gd)
print("LinearReg (GD) MSE:", mse(y_hat, y))

# Closed form (with bias)
Xsb = add_bias(Xs)
w_cf = linreg_closed_form(Xsb, y)
y_hat_cf = Xsb @ w_cf
print("LinearReg (Closed-form) MSE:", mse(y_hat_cf, y))

print(w_cf), print(w_gd)

LinearReg (GD) MSE: 0.08212219407255082
LinearReg (Closed-form) MSE: 0.08212219407255082
[ 1.99265209 -3.0734397   0.44669338  0.98758903]
[ 1.99265209 -3.0734397   0.44669338  0.98758903]


(None, None)

## 2) Logistic Regression (NumPy) — Binary Classification with GD
**Model**: \( p(y=1|x)=\sigma(Xw) \) (bias included).

In [ ]:
def logreg_fit(X, y, lr=0.1, steps=2000, l2=0.0):
    Xb = add_bias(X)
    n, d = Xb.shape
    y = y.reshape(-1)
    w = np.zeros(d)

    for _ in range(steps):
        logits = Xb @ w
        p = sigmoid(logits)
        grad = (Xb.T @ (p - y)) / n + 2*l2*w
        w -= lr * grad
    return w

def logreg_predict_proba(X, w):
    Xb = add_bias(X)
    return sigmoid(Xb @ w)

def logreg_predict(X, w, threshold=0.5):
    return (logreg_predict_proba(X, w) >= threshold).astype(int)

# Demo
seed_all(2)
n, d = 400, 2
X = np.random.randn(n, d)
true_w = np.array([1.5, -2.0])
true_b = 0.3
logits = X @ true_w + true_b
p = sigmoid(logits)
y = (np.random.rand(n) < p).astype(int)

mu, sigma = standardize_fit(X)
Xs = standardize_transform(X, mu, sigma)

w = logreg_fit(Xs, y, lr=0.3, steps=4000, l2=1e-4)
pred = logreg_predict(Xs, w)
print("LogReg acc:", accuracy(pred, y))
print("LogReg logloss:", log_loss(logreg_predict_proba(Xs, w), y))

## 3) Perceptron (NumPy) — Classic Linear Classifier
Labels \( y \in \{-1,+1\} \).

In [7]:
#rough cell
import numpy as np
temp0 = np.zeros(10)
temp1 = np.zeros([10,1])

temp0.shape, temp1.shape

((10,), (10, 1))

In [ ]:
def perceptron_fit(X, y, lr=1.0, epochs=20):
    # Grad update
    # If yi = +1 and we misclassified it, we push w toward xi (increase score for similar points).
    # If yi = -1, we push w away from xi (decrease score for similar points).
    Xb = add_bias(X)
    y = y.reshape(-1)
    w = np.zeros(Xb.shape[1])

    for _ in range(epochs):
        mistakes = 0
        for xi, yi in zip(Xb, y):
            if yi * (xi @ w) <= 0:
                w += lr * yi * xi
                mistakes += 1
        if mistakes == 0:
            break
    return w

def perceptron_predict(X, w):
    Xb = add_bias(X)
    return np.where((Xb @ w) >= 0, 1, -1)

# Demo
seed_all(3)
n = 200
X_pos = np.random.randn(n//2, 2) + np.array([2, 2])
X_neg = np.random.randn(n//2, 2) + np.array([-2, -2])
X = np.vstack([X_pos, X_neg])
y = np.array([1]*(n//2) + [-1]*(n//2))

mu, sigma = standardize_fit(X)
Xs = standardize_transform(X, mu, sigma)

w = perceptron_fit(Xs, y, lr=1.0, epochs=30)
pred = perceptron_predict(Xs, w)
print("Perceptron acc:", np.mean(pred == y))

## 4) K-Means (NumPy) — Clustering

In [ ]:
def kmeans_fit(X, k, steps=100, seed=42):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    centers = X[rng.choice(n, size=k, replace=False)]

    for _ in range(steps):
        dists = np.sum((X[:, None, :] - centers[None, :, :])**2, axis=2)
        labels = np.argmin(dists, axis=1)

        new_centers = np.zeros_like(centers)
        for j in range(k):
            mask = (labels == j)
            if np.any(mask):
                new_centers[j] = X[mask].mean(axis=0)
            else:
                new_centers[j] = X[rng.integers(0, n)]
        if np.allclose(new_centers, centers, atol=1e-6):
            centers = new_centers
            break
        centers = new_centers
    return centers, labels

def kmeans_inertia(X, centers, labels):
    return np.sum((X - centers[labels])**2)

# Demo (3 blobs)
seed_all(4)
n = 450
X = np.vstack([
    np.random.randn(n//3, 2) + np.array([0, 0]),
    np.random.randn(n//3, 2) + np.array([6, 0]),
    np.random.randn(n//3, 2) + np.array([3, 5]),
])

mu, sigma = standardize_fit(X)
Xs = standardize_transform(X, mu, sigma)

centers, labels = kmeans_fit(Xs, k=3, steps=100, seed=4)
print("KMeans inertia:", kmeans_inertia(Xs, centers, labels))
print("Centers:", centers)

## 5) PCA (NumPy) — Dimensionality Reduction via SVD

In [ ]:
def pca_fit(X, n_components):
    mu = X.mean(axis=0, keepdims=True)
    Xc = X - mu
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    components = Vt[:n_components]  # (k,d)
    explained_var = (S**2) / (X.shape[0] - 1)
    explained_ratio = explained_var[:n_components] / explained_var.sum()
    return mu, components, explained_ratio

def pca_transform(X, mu, components):
    return (X - mu) @ components.T

def pca_inverse_transform(Z, mu, components):
    return Z @ components + mu

# Demo
seed_all(5)
X = np.random.randn(500, 5)
X[:, 0] *= 5
X[:, 1] *= 3

mu, comps, ratio = pca_fit(X, n_components=2)
Z = pca_transform(X, mu, comps)
X_rec = pca_inverse_transform(Z, mu, comps)
print("PCA explained ratio:", ratio)
print("Reconstruction MSE (2D):", mse(X_rec, X))

## 6) Gaussian Naive Bayes (NumPy)

In [8]:
def gnb_fit(X, y):
    y = y.reshape(-1)
    classes = np.unique(y)
    params = {}
    for c in classes:
        Xc = X[y == c]
        mu = Xc.mean(axis=0)
        var = Xc.var(axis=0) + 1e-9
        prior = Xc.shape[0] / X.shape[0]
        params[c] = (mu, var, prior)
    return params

def gnb_predict(X, params):
    classes = sorted(params.keys())
    logps = []
    for c in classes:
        mu, var, prior = params[c]
        ll = -0.5*np.sum(np.log(2*np.pi*var) + ((X - mu)**2)/var, axis=1)
        logps.append(ll + np.log(prior + 1e-12))
    logps = np.stack(logps, axis=1)
    idx = np.argmax(logps, axis=1)
    return np.array([classes[i] for i in idx])

# Demo
seed_all(6)
n, d = 500, 4
X0 = np.random.randn(n//2, d)
X1 = np.random.randn(n//2, d) + 2.0
X = np.vstack([X0, X1])
y = np.array([0]*(n//2) + [1]*(n//2))

mu, sigma = standardize_fit(X)
Xs = standardize_transform(X, mu, sigma)

params = gnb_fit(Xs, y)
pred = gnb_predict(Xs, params)
print("GNB acc:", accuracy(pred, y))

GNB acc: 0.986


# PyTorch Section (Minimal, Interview-Style)

In [ ]:
import torch
from torch import nn

def seed_all_torch(seed=42):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all_torch(42)

def torch_train_loop(model, X, y, loss_fn, lr=1e-2, steps=500, optimizer_cls=torch.optim.SGD):
    model.train()
    opt = optimizer_cls(model.parameters(), lr=lr)
    for _ in range(steps):
        opt.zero_grad(set_to_none=True)
        y_hat = model(X)
        loss = loss_fn(y_hat, y)
        loss.backward()
        opt.step()
    return loss.item()

## 7) Linear Regression (PyTorch) — `nn.Linear`

In [ ]:
seed_all_torch(7)
n, d = 256, 3
X = torch.randn(n, d)
true_w = torch.tensor([2.0, -3.0, 0.5]).reshape(d,1)
true_b = torch.tensor([1.2])
y = X @ true_w + true_b + 0.3*torch.randn(n,1)

model = nn.Linear(d, 1)
loss_fn = nn.MSELoss()
final_loss = torch_train_loop(model, X, y, loss_fn, lr=0.1, steps=800, optimizer_cls=torch.optim.SGD)

with torch.no_grad():
    y_hat = model(X)
print("Torch LinReg loss:", final_loss)
print("Torch LinReg MSE:", torch.mean((y_hat - y)**2).item())

## 8) Logistic Regression (PyTorch) — `BCEWithLogitsLoss`

In [ ]:
seed_all_torch(8)
n, d = 400, 2
X = torch.randn(n, d)
true_w = torch.tensor([1.5, -2.0]).reshape(d,1)
true_b = torch.tensor([0.3])
logits = X @ true_w + true_b
p = torch.sigmoid(logits)
y = (torch.rand(n,1) < p).float()

model = nn.Linear(d, 1)
loss_fn = nn.BCEWithLogitsLoss()
final_loss = torch_train_loop(model, X, y, loss_fn, lr=0.2, steps=1200, optimizer_cls=torch.optim.SGD)

with torch.no_grad():
    probs = torch.sigmoid(model(X))
    pred = (probs >= 0.5).float()
acc = (pred == y).float().mean().item()
print("Torch LogReg loss:", final_loss)
print("Torch LogReg acc:", acc)

## 9) Perceptron (Torch-ish) — Manual Update (No autograd)

In [ ]:
seed_all_torch(9)
n = 200
X_pos = torch.randn(n//2, 2) + torch.tensor([2.0, 2.0])
X_neg = torch.randn(n//2, 2) + torch.tensor([-2.0, -2.0])
X = torch.cat([X_pos, X_neg], dim=0)
y = torch.cat([torch.ones(n//2), -torch.ones(n//2)], dim=0)  # {-1,+1}

Xb = torch.cat([X, torch.ones(n,1)], dim=1)
w = torch.zeros(3)

lr = 1.0
for epoch in range(30):
    mistakes = 0
    for i in range(n):
        if y[i] * (Xb[i] @ w) <= 0:
            w += lr * y[i] * Xb[i]
            mistakes += 1
    if mistakes == 0:
        break

with torch.no_grad():
    pred = torch.where((Xb @ w) >= 0, torch.tensor(1.0), torch.tensor(-1.0))
acc = (pred == y).float().mean().item()
print("Torch Perceptron acc:", acc)

# Appendix: Interview Ritual Checklist
1. State shapes
2. Define objective
3. Forward pass
4. Gradient/autograd
5. Update loop
6. Sanity check